# Student Score Prediction — Linear Regression

---

**Project:** ML with Scikit-Learn  
**Notebook:** 02 — Student Score Prediction  
**Dataset:** student_scores_csv.txt  
**Algorithm:** Multiple Linear Regression  
**Author:** Ather-ops  

---

## Objective

This notebook builds a complete end-to-end **student score prediction pipeline** using Scikit-Learn's `LinearRegression`. Given three behavioral features — study hours, sleep hours, and attendance — the model learns to predict a student's final exam score.

This is the second project in the ML with Scikit-Learn curriculum. The preprocessing concepts from Project 01 (House Price Prediction) are applied here on a different domain — education data.

---

## Pipeline Overview

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load dataset and inspect structure |
| 3 | Exploratory visualization — before training |
| 4 | Descriptive statistics and missing value check |
| 5 | Outlier detection using IQR |
| 6 | Feature and target selection + quantile bucketing |
| 7 | Train-test split |
| 8 | Feature scaling |
| 9 | Model training |
| 10 | Predictions |
| 11 | Model evaluation |
| 12 | Post-training visualization |
| 13 | Predict score for a new student |
| 14 | Final visualization — new student prediction in context |

---

## The Prediction Formula

Linear Regression learns a weighted equation:

$$\hat{score} = w_1 \times study\_hours + w_2 \times sleep\_hours + w_3 \times attendance + b$$

Where $w_1, w_2, w_3$ are learned weights and $b$ is the bias term. The model finds values for these that minimize prediction error across all training students.

---

## Dataset Description

| Column | Type | Range | Description |
|--------|------|-------|-------------|
| `study_hours` | float | 0 to 24 | Daily hours spent studying |
| `sleep_hours` | float | 0 to 12 | Daily hours of sleep |
| `attendance` | float | 0 to 100 | Class attendance percentage |
| `final_score` | float | 0 to 100 | Final exam score — target variable |

---
## Step 1 — Import Libraries

All required libraries are imported upfront. Each serves a specific role in the pipeline:

| Library | Role |
|---------|------|
| `pandas` | Load CSV, inspect DataFrame, quantile bucketing |
| `numpy` | Array creation for new student input |
| `matplotlib.pyplot` | All scatter plots and visualizations |
| `train_test_split` | Split data into 80% training and 20% test |
| `LinearRegression` | The regression model |
| `StandardScaler` | Normalize features to equal scale |
| `mean_squared_error` | Average squared prediction error |
| `mean_absolute_error` | Average absolute prediction error |
| `r2_score` | Proportion of score variance explained by the model |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

---
## Step 2 — Load Dataset

We load the student scores CSV and print the first 10 rows. This gives an immediate sense of the data structure, value ranges, and whether any columns look suspicious.

**What to check on first look:**
- Are all four expected columns present?
- Do study hours fall in a realistic range (e.g. 0 to 24)?
- Do final scores fall between 0 and 100?
- Are there any obviously wrong values visible in the first rows?

In [ ]:
# step 1: Load Data

df=pd.read_csv("student_scores_csv.txt")
print("original data :")
print("="*60)
print(df.head(10))
print("="*60)

---
## Step 3 — Initial Visualization — Before Training

Before building the model, we plot each feature against the target `final_score` to visually understand the relationships in the raw data.

**What each plot should show:**

| Plot | Expected trend | Reasoning |
|------|---------------|----------|
| Study Hours vs Final Score | Positive — more study, higher score | Core academic behavior |
| Sleep Hours vs Final Score | Mild positive — adequate sleep aids retention | Cognitive science basis |
| Attendance vs Final Score | Positive — more time in class, better outcomes | Exposure to material |

If a feature shows no clear trend with the target, it may have low predictive value. These scatter plots are the simplest and most informative first step of exploratory data analysis.

`plt.subplot(1,3,n)` arranges all three plots in a single row for easy comparison.

In [ ]:
# step 2: Initail visualisation
plt.figure(figsize=(15,4))
# 1:STUDY HOURS VS FINAL SCORE
plt.subplot(1,3,1)
plt.scatter(df["study_hours"],df["final_score"],color="red")
plt.xlabel("Study Hours")
plt.ylabel("Final Score")
plt.title("Study Hours VS Final Score(Before Training)")
plt.grid(True)
# 2: SLEPP HOURS VS FINAL SCORE
plt.subplot(1,3,2)
plt.scatter(df["sleep_hours"],df["final_score"],color="yellow")
plt.xlabel("Sleep Hours")
plt.ylabel("Final score")
plt.title("Sleep hours VS Final score(Before Training)")
plt.grid(True)
# 3: ATTENDANCE VS FINAL SCORE 
plt.subplot(1,3,3)
plt.scatter(df["attendance"],df["final_score"],color="green")
plt.xlabel("Attendance")
plt.ylabel("Final Score")
plt.title("Attendance VS Final score (Before Training)")
plt.tight_layout()
plt.grid(True)

---
## Step 4 — Descriptive Statistics and Missing Value Check

### Descriptive Statistics — `df.describe()`

`describe()` prints a statistical summary of every numerical column. Reading this before modeling is essential:

| Statistic | What to look for |
|-----------|------------------|
| `count` | If any column has fewer rows than others, it has missing values |
| `mean` | Should be realistic — e.g. mean study hours of 8–10 is plausible |
| `std` | High std relative to mean signals high spread or outliers |
| `min` | Negative study hours or scores below 0 would be data errors |
| `max` | Scores above 100 or study hours above 24 would be impossible values |
| `50%` (median) | Compare with mean — large gap indicates skew or outliers |

### Missing Values — `df.isnull().sum()`

Returns the count of `NaN` values per column. Any non-zero value must be addressed before training — `LinearRegression` raises a `ValueError` if the input contains `NaN`.

In [ ]:
# step 3: Data expoloration(EDA)
print("Basic statistic:",df.describe())
print("="*80)
print("missing values:",df.isnull().sum)
print("="*80)

---
## Step 5 — Outlier Detection using IQR

We apply the **Interquartile Range (IQR)** method to detect outliers in the `study_hours` column.

**Why `study_hours` specifically?**  
Study hours is the most impactful feature for predicting final score. An outlier here — for example a student logged with 23 study hours per day — would distort the regression line significantly.

**IQR formula:**

$$IQR = Q3 - Q1$$

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$

$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

| Variable | Meaning |
|----------|---------|
| Q1 | 25th percentile — bottom quarter of study hours |
| Q3 | 75th percentile — top quarter begins here |
| IQR | Width of the middle 50% of the distribution |
| Lower Bound | Below this = unusually low study hours |
| Upper Bound | Above this = unusually high study hours |

The printed `outliers` DataFrame shows every student whose study hours fall outside these boundaries. In a production pipeline these rows would be clipped or removed before training.

In [ ]:
# step 4: Finding outliers
Q1=df["study_hours"].quantile(0.25)
Q3=df["study_hours"].quantile(0.75)

IQR=Q3-Q1

lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR

outliers=df[(df["study_hours"]<lower_bound)|(df["study_hours"]>upper_bound)]
print("outlier:",outliers)

print("="*80)

---
## Step 6 — Feature Selection and Quantile Bucketing

### Feature and Target Selection

We define `X` as the three input features and `Y` as the target:

| Variable | Columns | Description |
|----------|---------|-------------|
| `X` | study_hours, sleep_hours, attendance | Input matrix — what the model learns from |
| `Y` | final_score | Target vector — what the model predicts |

### Quantile Bucketing — `pd.qcut()`

Unlike `pd.cut()` which uses fixed boundary values, **`pd.qcut()`** divides data into bins with **equal numbers of rows** in each bucket — this is called quantile-based bucketing.

With `q=3`, the study_hours column is divided into three equal-sized groups:

| Quantile | Label | Meaning |
|----------|-------|---------|
| Bottom 33% | `Q1_small` | Low study hours |
| Middle 33% | `Q2_medium` | Moderate study hours |
| Top 33% | `Q3_large` | High study hours |

**`pd.qcut()` vs `pd.cut()`:**

| Method | Boundary logic | Use when |
|--------|---------------|----------|
| `pd.cut()` | Fixed value boundaries | You know domain boundaries (e.g. 0-1000 sqft) |
| `pd.qcut()` | Equal-frequency boundaries | You want equal representation per group |

The `student_quantile` column is printed for inspection — it is used for analysis, not directly as a model feature in this pipeline.

In [ ]:
# step 5: TARGET AND FEATURE
X=df[["study_hours","sleep_hours","attendance"]]
Y=df["final_score"]
# Quantile buckting
df["student_quantile"]=pd.qcut(df["study_hours"],q=3,labels=["Q1_small","Q2_medium","Q3_large"])
print("Data after quantile bucting:")
print("="*80)
print(df.head(10))
print("="*80)

---
## Step 7 — Train-Test Split

We split the dataset into training and test sets. The model is trained only on `X_train` and `Y_train`. `X_test` and `Y_test` are held back and used only for final evaluation.

| Parameter | Value | Effect |
|-----------|-------|--------|
| `test_size=0.2` | 20% | 20% of students go to the test set |
| `random_state=42` | 42 | Fixed random seed — identical split on every run |

**Why this split is critical:**  
If we train and evaluate on the same data, the model appears to perform perfectly — it has already memorized those students. Evaluating on an unseen test set gives an honest measure of how well the model will perform on students it has never seen before.

**Resulting arrays:**

| Array | Contains |
|-------|----------|
| `X_train` | 80% of student feature rows — used for training |
| `X_test` | 20% of student feature rows — used for evaluation |
| `Y_train` | Corresponding final scores for training rows |
| `Y_test` | Corresponding final scores for test rows |

In [ ]:
#step 6: TRAIN TEST SPLIT
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

---
## Step 8 — Feature Scaling

The three features have very different natural scales:

| Feature | Typical range | Problem without scaling |
|---------|--------------|------------------------|
| `study_hours` | 0 to 24 | Medium magnitude |
| `sleep_hours` | 0 to 12 | Smaller magnitude |
| `attendance` | 0 to 100 | Largest magnitude — dominates the model |

Without scaling, `attendance` (0 to 100) contributes much larger raw values than `sleep_hours` (0 to 12). The model assigns disproportionate weight to attendance purely because of its numeric scale — not because it is more important.

**StandardScaler** transforms each feature to:
$$z = \frac{x - \mu}{\sigma}$$

After scaling, all three features have mean = 0 and standard deviation = 1 — they contribute equally to the model.

**The golden rule of scaling:**

| Call | Data | Why |
|------|------|-----|
| `fit_transform(X_train)` | Training only | Learns $\mu$ and $\sigma$ from training students |
| `transform(X_test)` | Test only | Applies the same $\mu$ and $\sigma$ — no new learning |

In [ ]:
# step 7:Feature scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

---
## Step 9 — Model Training

We initialize a `LinearRegression` model and train it on the scaled training data.

**What `.fit()` does internally:**

Scikit-Learn solves the Normal Equation to find the optimal weights:

$$\theta = (X^T X)^{-1} X^T Y$$

This gives the exact weights $w_1, w_2, w_3$ and bias $b$ that minimize the sum of squared errors across all training students — in one computation, no loop needed.

**After training, the model stores:**
- `model.coef_` — three weights, one per feature (study_hours, sleep_hours, attendance)
- `model.intercept_` — the bias term

A positive coefficient means that feature pushes the predicted score up. A negative coefficient means it pushes the score down.

In [ ]:
# step 8: Model selection
model=LinearRegression()
model.fit(X_train_scaled,Y_train)

---
## Step 10 — Predictions

We run the trained model on the scaled test features to generate predicted scores for every student in the test set.

**What happens during `model.predict()`:**

For each test student, the model computes:

$$\hat{score} = w_1 \times study\_hours_{scaled} + w_2 \times sleep\_hours_{scaled} + w_3 \times attendance_{scaled} + b$$

The result `y_pred` is a NumPy array of predicted final scores — one per test student. These predictions are then compared against `Y_test` (the actual scores) to evaluate model quality.

In [ ]:
# step 9: Predictions
y_pred=model.predict(X_test_scaled)

---
## Step 11 — Model Evaluation

We evaluate the trained model using three standard regression metrics:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ | Average squared error — penalizes large mistakes heavily |
| MAE | $\frac{1}{n}\sum|y - \hat{y}|$ | Average absolute error — easy to interpret in score units |
| R2 Score | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of score variance explained by the model |

**Interpreting R2 Score:**

| R2 Value | Meaning |
|----------|---------|
| 1.00 | Perfect predictions — model explains all variance |
| 0.90+ | Excellent — strong linear relationship captured |
| 0.70 – 0.90 | Good — model is useful but some variance unexplained |
| Below 0.50 | Weak — model is not capturing the relationship well |
| 0.00 | Model is no better than predicting the mean score for everyone |

For a student score dataset with three behavioral features, an R2 above 0.80 is a strong result.

In [ ]:
# stepn 10: Evalution
MSE=mean_squared_error(Y_test,y_pred)
MAE=mean_absolute_error(Y_test,y_pred)
R2_SCORE=r2_score(Y_test,y_pred)
print(f"MSE:{MSE:.3F}, MAE:{MAE:.3F},R2_SCORE:{R2_SCORE:.3F}")
print("="*80)

---
## Step 12 — Post-Training Visualization

We now plot the same three feature-vs-score relationships from Step 3 — but this time overlaying the model's predicted scores as a second scatter layer.

**How to read these plots:**

| Element | Meaning |
|---------|--------|
| First color dots (actual) | Real final scores from the test set |
| Second color dots (predicted) | What the model predicted for those same students |
| Tight overlap between colors | High accuracy — predictions are close to reality |
| Wide separation between colors | High error — model is struggling with that feature |

**What changes from the before-training plots:**
- Before training: only raw data visible, no model output
- After training: both actual and predicted scores shown side by side

Comparing these directly reveals where the model is accurate and where it over- or under-predicts.

In [ ]:
# step 11:Visuallisation after predictions
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
# 1: STUDY HOURS VS FINAL SCORE
plt.scatter(X_test["study_hours"],Y_test,label="Actual line",color="pink")
plt.scatter(X_test["study_hours"],y_pred,label="Predicted line",color="green")
plt.title(" STUDY HOURS VS FINAL SCORE(After Training)")
plt.xlabel("STUDY HOURS")
plt.ylabel("FINAL SCORE")
plt.grid(True)
plt.legend()
# 2: SLEEP HOURS VS FINAL SCORE
plt.subplot(1,3,2)
plt.scatter(X_test["sleep_hours"],Y_test,color="black",label="Actual line")
plt.scatter(X_test["sleep_hours"],y_pred,label="Predicted line",color="blue")
plt.xlabel("SLEEP HOURS")
plt.ylabel("FINAL SCORE")
plt.title("SLEPP HOURS VS FINAL SCORE (After Training)")
plt.grid(True)
# 3: ATTENDANCE VS FINAL SCORE
plt.subplot(1,3,3)
plt.scatter(X_test["attendance"],Y_test,color="red",label="Actual line")
plt.scatter(X_test["attendance"],y_pred,label="Predicted line",color="orange")
plt.xlabel("ATTENDANCE")
plt.ylabel("FINAL SCORE")
plt.title("ATTENDANCE VS FINAL SCORE (After Training)")
plt.grid(True)
plt.tight_layout()
print("="*100)

---
## Step 13 — Predict Score for a New Student

This is the real-world use of the trained model. Given a student the model has never seen, we predict their expected final score.

**New student profile:**

| Feature | Value | Context |
|---------|-------|--------|
| `study_hours` | 14.5 | Above average — serious student |
| `sleep_hours` | 6 | Slightly below recommended |
| `attendance` | 76 | Decent but not exceptional |

**Prediction pipeline for new input:**

1. Create input as a NumPy array with shape `(1, 3)` — one row, three features
2. Scale using the **same scaler** fitted on training data — `scaler.transform()` only, never `fit_transform()`
3. Pass through `model.predict()` to get the predicted score

> Using `scaler.transform()` (not `fit_transform()`) on new input is critical. The scaler must apply the same mean and std it learned from the training set — not recompute new statistics from this single row.

In [ ]:
# step 12: Predicting new values 
new_student=np.array([[14.5,6,76]])
new_student_scaled=scaler.transform(new_student)
predicted_score=model.predict(new_student_scaled)
print(f"predicted new score of new student is :{predicted_score[0]:.2f}")
print("="*100)

---
## Step 14 — Final Visualization — New Student in Context

Now that we have the predicted score for our new student, we place that prediction in the context of the full test set. This answers the question visually: where does this new student sit relative to all other students the model has seen?

**Three panels:**

| Panel | X-axis | What the star marks |
|-------|--------|---------------------|
| 1 | Study Hours | New student at study_hours = 14.5 |
| 2 | Sleep Hours | New student at sleep_hours = 6 |
| 3 | Attendance | New student at attendance = 76 |

The large red star on each plot shows exactly where the new student falls — both their feature value on the x-axis and their predicted score on the y-axis.

A final **Actual vs Predicted** scatter plot and **Residual Distribution** histogram round out the diagnostic view of overall model performance.

In [ ]:
# step 14: Final visualization — new student prediction in context

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Study Hours
axes[0].scatter(X_test["study_hours"], Y_test,    color="pink",   label="Actual",    alpha=0.7)
axes[0].scatter(X_test["study_hours"], y_pred,    color="green",  label="Predicted", alpha=0.7)
axes[0].scatter(14.5, predicted_score[0],
                color="red", marker="*", s=300, zorder=5, label="New Student")
axes[0].set_xlabel("Study Hours")
axes[0].set_ylabel("Final Score")
axes[0].set_title("Study Hours — New Student Highlighted")
axes[0].legend()
axes[0].grid(True)

# Panel 2: Sleep Hours
axes[1].scatter(X_test["sleep_hours"], Y_test,    color="black",  label="Actual",    alpha=0.7)
axes[1].scatter(X_test["sleep_hours"], y_pred,    color="blue",   label="Predicted", alpha=0.7)
axes[1].scatter(6, predicted_score[0],
                color="red", marker="*", s=300, zorder=5, label="New Student")
axes[1].set_xlabel("Sleep Hours")
axes[1].set_ylabel("Final Score")
axes[1].set_title("Sleep Hours — New Student Highlighted")
axes[1].legend()
axes[1].grid(True)

# Panel 3: Attendance
axes[2].scatter(X_test["attendance"], Y_test,     color="red",    label="Actual",    alpha=0.7)
axes[2].scatter(X_test["attendance"], y_pred,     color="orange", label="Predicted", alpha=0.7)
axes[2].scatter(76, predicted_score[0],
                color="blue", marker="*", s=300, zorder=5, label="New Student")
axes[2].set_xlabel("Attendance")
axes[2].set_ylabel("Final Score")
axes[2].set_title("Attendance — New Student Highlighted")
axes[2].legend()
axes[2].grid(True)

plt.suptitle(
    f"New Student Predicted Score: {predicted_score[0]:.2f}  |  Study: 14.5h  Sleep: 6h  Attendance: 76%",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()
print("="*100)

---
## Step 15 — Actual vs Predicted and Residual Analysis

Two final diagnostic plots to assess overall model quality:

### Actual vs Predicted Scatter

Plots actual test scores on the X-axis against predicted scores on the Y-axis:
- A perfect model produces all points on the diagonal line `y = x`
- Points above the diagonal = model underestimated the score
- Points below the diagonal = model overestimated the score
- Tight clustering around the diagonal = high R2 score

### Residual Distribution

Residuals = Actual Score minus Predicted Score:

| Residual Pattern | Meaning |
|-----------------|--------|
| Bell curve centered at 0 | Errors are random and unbiased — ideal |
| Skewed left or right | Model is systematically over- or under-predicting |
| Very wide spread | High variance — model is uncertain on many students |

A good linear regression model should produce residuals normally distributed around zero. If the residuals show a pattern (e.g. all positive for high scorers), it suggests a non-linear relationship that a linear model cannot fully capture.

In [ ]:
# step 15: Actual vs Predicted and Residual Distribution
residuals = Y_test - y_pred

plt.figure(figsize=(12, 4))

# Plot 1: Actual vs Predicted
plt.subplot(1, 2, 1)
plt.scatter(Y_test, y_pred, color="steelblue", alpha=0.7, edgecolors="white", linewidth=0.5)
min_val = min(Y_test.min(), y_pred.min())
max_val = max(Y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val],
         color="red", linestyle="--", linewidth=1.5, label="Perfect prediction")
plt.xlabel("Actual Score")
plt.ylabel("Predicted Score")
plt.title("ACTUAL VS PREDICTED SCORE")
plt.legend()
plt.grid(True)

# Plot 2: Residual Distribution
plt.subplot(1, 2, 2)
plt.hist(residuals, bins=15, color="steelblue", edgecolor="white")
plt.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Zero error line")
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Frequency")
plt.title("RESIDUAL DISTRIBUTION")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
print("="*80)

---
## Pipeline Summary

This notebook built a complete student score prediction pipeline from raw CSV data to a deployed model with new-value inference.

| Step | Action | Output |
|------|--------|--------|
| 1 | Import libraries | All tools ready |
| 2 | Load dataset | Raw DataFrame with 4 columns |
| 3 | EDA scatter plots | Visual understanding of feature-score relationships |
| 4 | Descriptive statistics and missing value check | Data quality confirmed |
| 5 | IQR outlier detection on study_hours | Outlier rows identified |
| 6 | Feature selection and quantile bucketing | X, Y defined; student_quantile column added |
| 7 | Train-test split (80/20) | X_train, X_test, Y_train, Y_test |
| 8 | StandardScaler — fit on train, transform both | X_train_scaled, X_test_scaled |
| 9 | LinearRegression model trained | Weights and bias learned |
| 10 | Predictions on test set | y_pred array |
| 11 | Evaluation — MSE, MAE, R2 | Quantified model performance |
| 12 | Post-training scatter plots | Actual vs predicted per feature |
| 13 | New student prediction | Single score predicted from 3 inputs |
| 14 | New student visualization | Star-marked position across all three feature plots |
| 15 | Actual vs Predicted and Residual plot | Model diagnostics complete |

---

**Next notebook:** `03_logistic_regression.ipynb` — Binary Classification